# A tour of the U.S. Treasury yield curve

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RblxDev-ALS/NSS-Yield-Curve-Engine/blob/main/examples/tour.ipynb)

This notebook uses [nss-engine](https://github.com/RblxDev-ALS/NSS-Yield-Curve-Engine) to answer
four questions with live Treasury data from FRED (no API key needed):

1. **What does the yield curve look like today?** Fit the Nelson–Siegel–Svensson model.
2. **Is the curve signalling a recession?** The 10-year minus 3-month spread and a probit model.
3. **Why are long rates where they are?** Split the 10-year yield into expected short rates and a term premium.
4. **What would a bond portfolio lose if rates rose?** Duration, DV01 and key-rate risk.

Run the cells top to bottom (Runtime → Run all). It takes about three minutes.

In [ ]:
# Install the engine (in Colab; skip if already installed)
%pip install -q git+https://github.com/RblxDev-ALS/NSS-Yield-Curve-Engine.git

In [ ]:
import numpy as np
import plotly.graph_objects as go

from nss_engine import calibrate, calibrate_panel
from nss_engine.data import load_recession_indicator, load_treasury_yields

SOURCE = "fred"  # set to "synthetic" to run offline on a simulated market

if SOURCE == "fred":
    yields = load_treasury_yields(start="1990-01-01")  # weekly, columns = maturities in years
    recession = load_recession_indicator(start="1990-01-01")
else:
    from nss_engine.synthetic import simulate_market

    mkt = simulate_market(seed=0)
    yields, recession = mkt.yields, mkt.recession
yields.tail()

## 1. Today's curve

The market quotes 11 yields, from 1 month to 30 years. The NSS model turns them into a smooth
curve with six parameters: a long-run **level** (β0), a **slope** (β1), and two **humps**.
The quotes are *par yields*, so the engine fits them as such and returns the zero-coupon curve.

In [ ]:
date, quotes = yields.index[-1], yields.iloc[-1].dropna()
fit = calibrate(np.asarray(quotes.index, dtype=float), quotes.to_numpy())
curve = fit.curve
print(f"{date:%d %b %Y}: fit error {fit.rmse_bp:.1f} bp")
print(
    f"level {curve.beta0:.2f}%, short rate {curve.short_rate:.2f}%, "
    f"10y-3m spread {curve.spread(10, 0.25, 'par'):+.2f} pp"
)

grid = np.linspace(1 / 12, 30, 200)
lo, hi = fit.confidence_band(grid)
fig = go.Figure()
fig.add_scatter(
    x=np.r_[grid, grid[::-1]],
    y=np.r_[hi, lo[::-1]],
    fill="toself",
    line_width=0,
    name="95% band (zero)",
    opacity=0.25,
)
fig.add_scatter(x=grid, y=curve.zero(grid), name="zero curve")
fig.add_scatter(x=grid, y=curve.forward(grid), name="forward curve", line_dash="dot")
fig.add_scatter(x=quotes.index, y=quotes, mode="markers", name="quotes (par)")
fig.update_layout(
    title=f"U.S. Treasury curve, {date:%d %b %Y}", xaxis_title="maturity (years)", yaxis_title="%"
)
fig.show()

## 2. Recession signal

Fitting every week since 1990 (about a minute) gives the history of the curve. An inverted curve
(10-year below 3-month) has preceded every U.S. recession since the 1960s. A probit model turns
the spread into a probability of recession 12 months ahead, the New York Fed's specification.

In [ ]:
from nss_engine.regime import recession_probability_model

panel = calibrate_panel(yields)
spread = panel.spread(10, 0.25)  # model-implied 10y - 3m, percentage points
model = recession_probability_model(spread, recession, horizon=12)
print(f"Probability of recession in {model.target_date:%b %Y}: {model.latest_probability:.0%}")
print(f"(in-sample AUC {model.auc:.2f}; four recessions since 1990, so treat as indicative)")

fig = go.Figure()
fig.add_scatter(x=spread.index, y=spread, name="10y - 3m spread (pp)")
fig.add_scatter(x=model.fitted.index, y=model.fitted * 10, name="recession prob. x10", yaxis="y")
for start, end in (
    recession.groupby((recession != recession.shift()).cumsum())
    .apply(lambda s: (s.index[0], s.index[-1]) if s.iloc[0] == 1 else None)
    .dropna()
):
    fig.add_vrect(x0=start, x1=end, fillcolor="grey", opacity=0.2, line_width=0)
fig.add_hline(y=0)
fig.update_layout(title="Slope of the curve and recession probability (shaded: NBER recessions)")
fig.show()

## 3. Expected rates vs term premium

A 10-year yield = the average short rate investors expect over ten years + a **term premium**
(extra pay for holding a long bond). The Adrian–Crump–Moench model, the method behind the
New York Fed's published premium, separates the two using only regressions.

In [ ]:
from nss_engine.termpremium import fit_acm, zero_panel

acm = fit_acm(zero_panel(panel.params))  # month-end zero curves, 1-120 months
dec = acm.decomposition(10)
last = dec.iloc[-1]
print(
    f"10y zero yield {last['fitted']:.2f}% = expected short rate "
    f"{last['expected_short_rate']:.2f}% + term premium {last['term_premium']:+.2f}%"
)

fig = go.Figure()
for col, name in [
    ("fitted", "10y yield"),
    ("expected_short_rate", "expected short rate"),
    ("term_premium", "term premium"),
]:
    fig.add_scatter(x=dec.index, y=dec[col], name=name)
fig.update_layout(title="Decomposing the 10-year yield", yaxis_title="%")
fig.show()

## 4. Bond risk

How much does a 10-year Treasury lose if rates rise? Duration and DV01 answer for a parallel
shift; key-rate durations show which part of the curve the bond is exposed to.

In [ ]:
from nss_engine.analytics import Bond, risk_report

bond = Bond.par(curve, 10.0)
rep = risk_report(curve, bond)
print(
    f"10y par bond: price {rep.price:.2f}, duration {rep.duration:.2f} years, "
    f"DV01 ${rep.dv01 * 10_000:,.0f} per $1m face for a 1 bp move"
)
print(f"A 1-point rise in all rates costs about {rep.duration:.1f}% of the price.")
rep.key_rate_durations.round(3)

## Where to go next

* `nss-engine run` builds the full interactive dashboard and a Markdown report.
* The [README](https://github.com/RblxDev-ALS/NSS-Yield-Curve-Engine) lists the validation results:
  agreement with the Federal Reserve's own curve, pseudo-real-time recession tests, and honest
  forecast comparisons against the random walk.
* Educational project, not investment advice.